In [14]:
import pandas as pd

gtfs = "/home/maciek/Projects/wiki/wd/mpk_lodz/GTFS"

# Load data
stops = pd.read_csv(f"{gtfs}/stops.txt", encoding="utf-8-sig", dtype={"stop_code": str})
stops["stop_code"] = stops["stop_code"].str.strip()
routes = pd.read_csv(f"{gtfs}/routes.txt", encoding="utf-8-sig")
trips = pd.read_csv(f"{gtfs}/trips.txt", encoding="utf-8-sig", usecols=["trip_id", "route_id"])

# Build trip -> route_type lookup
trip_route = trips.merge(routes[["route_id", "route_type"]], on="route_id")
trip_type_map = dict(zip(trip_route["trip_id"], trip_route["route_type"]))

# Classify each stop
stop_types = {}
for chunk in pd.read_csv(f"{gtfs}/stop_times.txt", encoding="utf-8-sig",
                          usecols=["trip_id", "stop_id"], chunksize=100000):
    for _, row in chunk.iterrows():
        rt = trip_type_map.get(row["trip_id"])
        if rt is not None:
            stop_types.setdefault(row["stop_id"], set()).add(rt)

# Add classification column to stops
def classify(stop_id):
    types = stop_types.get(stop_id, set())
    if types == {0}:
        return "tram"
    elif types == {3}:
        return "bus"
    elif types == {0, 3}:
        return "both"
    return "unknown"

stops["stop_type"] = stops["stop_id"].apply(classify)

# Save enriched stops
stops.to_csv(f"{gtfs}/stops_classified.csv", index=False, encoding="utf-8")
print(stops["stop_type"].value_counts())
stops.head()


stop_type
bus     1908
tram     375
both     117
Name: count, dtype: int64


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,stop_type
0,1,0001,Abramowskiego-Sienkiewicza,NaN,51.75424,19.46545,1,NaN,0,NaN,NaN,0,bus
1,2,0081,Brzeźna-Piotrkowska,NaN,51.75286,19.46029,1,NaN,0,NaN,NaN,0,bus
2,3,0002,Kochanówka,NaN,51.80451,19.35784,1,NaN,0,NaN,NaN,0,tram
3,4,0003,Aleksandrowska-Lechicka,NaN,51.80356,19.36354,1,NaN,0,NaN,NaN,0,tram
4,5,0004,Aleksandrowska-Szczecińska,NaN,51.80279,19.36713,1,NaN,0,NaN,NaN,0,tram


In [42]:
import pandas as pd
import re
from math import radians, sin, cos, sqrt, atan2

gtfs = "/home/maciek/Projects/wiki/wd/mpk_lodz/GTFS"
stops = pd.read_csv(f"{gtfs}/stops_classified.csv", encoding="utf-8", dtype={"stop_code": str})
groups = pd.read_csv("/home/maciek/Projects/wiki/wd/mpk_lodz/groups_of_stops.tsv", sep="\t")

# Parse group coordinates and Q-ids
def parse_point(p):
    m = re.search(r'Point\(([\d.]+)\s+([\d.]+)\)', str(p))
    return (float(m.group(2)), float(m.group(1))) if m else (None, None)

groups["g_lat"], groups["g_lon"] = zip(*groups["coordinates"].apply(parse_point))
groups["qid"] = groups["item"].str.extract(r'(Q\d+)')

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def normalize(name):
    return name.strip('"').lower().replace(' ','').replace('-','')

# Build stop_id -> group_qid mapping
# Build stop_id -> (group_qid, adm_qid) mapping
stop_group = {}
stop_adm = {}
for _, stop in stops.iterrows():
    sname = stop["stop_name"].strip('"')
    snorm = normalize(sname)
    best, best_dist, best_adm = None, float('inf'), None
    for _, grp in groups.iterrows():
        gnorm = normalize(grp["itemLabel"])
        if gnorm in snorm or snorm in gnorm:
            d = haversine(stop["stop_lat"], stop["stop_lon"], grp["g_lat"], grp["g_lon"])
            if d < best_dist:
                best, best_dist = grp["qid"], d
                # Extract Q-id from adm URL
                adm_match = re.search(r'(Q\d+)', str(grp["adm"]))
                best_adm = adm_match.group(1) if adm_match else None
    if best and best_dist < 1000:
        stop_group[stop["stop_id"]] = best
        if best_adm:
            stop_adm[stop["stop_id"]] = best_adm

# --- QuickStatements generation ---
type_qids = {
    "bus":  ["Q953806"],
    "tram": ["Q2175765"],
    "both": ["Q953806", "Q2175765"],
}
system_qids = {
    "bus":  ["Q9162523"],
    "tram": ["Q1149987"],
    "both": ["Q9162523", "Q1149987"],
}
source = '\tS854\t"https://otwarte.miasto.lodz.pl/transport_komunikacja/"\tS813\t+2026-03-29T00:00:00Z/11'

lines = []
for _, row in stops.iterrows():
    name = row["stop_name"].strip('"')
    code = row["stop_code"]
    lat, lon = row["stop_lat"], row["stop_lon"]

    lines.append("CREATE")
    lines.append(f'LAST\tLpl\t"{name} ({code})"')
    lines.append(f'LAST\tDpl\t"przystanek komunikacji miejskiej w aglomeracji łódzkiej"')
    lines.append(f'LAST\tApl\t"{code}"')

    for qid in type_qids.get(row["stop_type"], ["Q953806"]):
        lines.append(f'LAST\tP31\t{qid}{source}')
    for qid in system_qids.get(row["stop_type"], ["Q9162523"]):
        lines.append(f'LAST\tP16\t{qid}{source}')
    if " NŻ" in name:
        lines.append(f'LAST\tP31\tQ813817{source}')
    lines.append(f'LAST\tP137\tQ11780295\t{source}')
    lines.append(f'LAST\tP5817\tQ55654238\t{source}')
    lines.append(f'LAST\tP17\tQ36')
    
    adm_qid = stop_adm.get(row["stop_id"], "Q580")
    lines.append(f'LAST\tP131\t{adm_qid}{source}')

    lines.append(f'LAST\tP625\t@{lat}/{lon}{source}')

    # P361 - part of group
    grp_qid = stop_group.get(row["stop_id"])
    if grp_qid:
        lines.append(f'LAST\tP361\t{grp_qid}{source}')
        lines.append(f'{grp_qid}\tP527\tLAST{source}')

    lines.append("")

output = "\n".join(lines)
with open("/home/maciek/Projects/wiki/wd/mpk_lodz/quickstatements_stops.txt", "w", encoding="utf-8") as f:
    f.write(output)

matched = sum(1 for s in stops["stop_id"] if s in stop_group)
print(f"Generated {len(stops)} items ({matched} with P361 group link)")
print("\n--- Preview (matched stop) ---")
# Show a stop that has P361
for line in output.split("\n"):
    if "P361" in line:
        # find the CREATE block
        idx = output.index(line)
        block_start = output.rfind("CREATE", 0, idx)
        block_end = output.index("\n\n", idx)
        print(output[block_start:block_end])
        break


Generated 2400 items (52 with P361 group link)

--- Preview (matched stop) ---
CREATE
LAST	Lpl	"Jaracza-P.O.W. (0223)"
LAST	Dpl	"przystanek komunikacji miejskiej w aglomeracji łódzkiej"
LAST	Apl	"0223"
LAST	P31	Q953806	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P16	Q9162523	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P137	Q11780295		S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P5817	Q55654238		S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P17	Q36
LAST	P131	Q9257542	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P625	@51.7737/19.46705	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P361	Q115234267	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
Q1

In [ ]:
import pandas as pd

gtfs = "/home/maciek/Projects/wiki/wd/mpk_lodz/GTFS"
routes = pd.read_csv(f"{gtfs}/routes.txt", encoding="utf-8-sig")

source = '\tS854\t"https://otwarte.miasto.lodz.pl/transport_komunikacja/"\tS813\t+2026-03-29T00:00:00Z/11'

# route_type: 0 = tram, 3 = bus
# P31: Q3240003 (bus route), Q15145593 (tram route/line)
lines = []
for _, row in routes.iterrows():
    name = row["route_short_name"].strip('"')
    is_tram = row["route_type"] == 0
    
    type_label = "tramwajowa" if is_tram else "autobusowa"
    p31 = "Q15145593" if is_tram else "Q3240003"
    
    label = f"linia nr {type_label} {name} w Łodzi"
    desc = f"linia {type_label} w aglomeracji łódzkiej"
    alias = name
    
    lines.append("CREATE")
    lines.append(f'LAST\tLpl\t"{label}"')
    lines.append(f'LAST\tDpl\t"{desc}"')
    lines.append(f'LAST\tP31\t{p31}{source}')
    lines.append(f'LAST\tP17\tQ36{source}')
    lines.append(f'LAST\tP131\tQ580{source}')
    lines.append("")

output = "\n".join(lines)

with open("/home/maciek/Projects/wiki/wd/mpk_lodz/quickstatements_routes.txt", "w", encoding="utf-8") as f:
    f.write(output)

print(f"Generated {len(routes)} route items ({sum(routes.route_type==0)} tram, {sum(routes.route_type==3)} bus)")
print("\n--- Preview ---")
print("\n".join(output.split("\n")[:16]))


Generated 133 route items (24 tram, 109 bus)

--- Preview ---
CREATE
LAST	Lpl	"linia nr tramwajowa 1 w Łodzi"
LAST	Dpl	"linia tramwajowa w aglomeracji łódzkiej"
LAST	P31	Q15145593	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P17	Q36	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P131	Q580	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11

CREATE
LAST	Lpl	"linia nr tramwajowa 10A w Łodzi"
LAST	Dpl	"linia tramwajowa w aglomeracji łódzkiej"
LAST	P31	Q15145593	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P17	Q36	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11
LAST	P131	Q580	S854	"https://otwarte.miasto.lodz.pl/transport_komunikacja/"	S813	+2026-03-29T00:00:00Z/11

CREATE
LAST	Lpl	"linia nr tramwajowa 10B w Łodzi"
